In [2]:
!pip install azure-storage-blob

^C
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.5 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.5 MB ? eta -:--:--
   ------ --------------------------------- 0.5/3.5 MB 560.1 kB/s eta 0:00:06
   ------ --------------------------------- 0.5/3.5 MB 560.1 kB/s eta 0:00:06
   --------- ------------------------------ 0.8/3.5 MB 671.3 kB/s eta 0:00:05
   ------------ --------------------------- 1.0/3.5 MB 680.3 kB/s eta 0:00:04
   ------------ --------------------------- 1.0/3.5 MB 680.3 kB/s eta 0:00:04
   --------------- ------------------------ 1.3/3.5 MB 729.7 kB/s eta 0:00:03
   ------------------ --------------------- 1.6/3.5 MB 749.0 kB/s eta 0:00:03
   --------------------- ------------------ 1.8/3.5 MB 792.8 kB/s eta 0:00:03
   ------------------------ 

In [3]:
import os
import json
import random
from tqdm import tqdm
from azure.storage.blob import ContainerClient

# Paths
JSON_PATH    = r'C:\Users\dippe\Desktop\University of Botswana\Semester 8\FINALYEARPROJECT\Datset\Serenegetijson\SnapshotSerengetiS03.json'
IMAGES_DIR   = r'C:\Users\dippe\Desktop\University of Botswana\Semester 8\PROJECT\data'

print('Loading JSON...')
with open(JSON_PATH, 'r') as f:
    data = json.load(f)
print(f'JSON loaded. Total annotations: {len(data["annotations"])}')

Loading JSON...
JSON loaded. Total annotations: 398689


In [2]:
import sys
!{sys.executable} -m pip install azure-storage-blob

  Using cached azure_storage_blob-12.28.0-py3-none-any.whl.metadata (26 kB)
Using cached azure_storage_blob-12.28.0-py3-none-any.whl (431 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Get empty frame annotations
empty_annotations = [ann for ann in data['annotations'] if ann['category_id'] == 0]
print(f'Total empty annotations: {len(empty_annotations)}')

# Sample 3000
random.seed(42)
sampled_empty = random.sample(empty_annotations, 3000)

# Build download list
to_download = []
for ann in sampled_empty:
    img_id    = ann['image_id']
    filename  = img_id.split('/')[-1] + '.JPG'
    blob_path = f'{img_id}.JPG'
    save_path = os.path.join(IMAGES_DIR, filename)
    to_download.append((blob_path, save_path))

print(f'Preparing to download {len(to_download)} images...')

# Connect to Azure
client = ContainerClient(
    account_url="https://lilawildlife.blob.core.windows.net",
    container_name="lila-wildlife",
    credential=None
)

success = 0
failed  = 0

for blob_path, save_path in tqdm(to_download):
    if os.path.exists(save_path):
        success += 1
        continue
    try:
        blob_client = client.get_blob_client(f'snapshotserengeti-unzipped/{blob_path}')
        with open(save_path, 'wb') as f:
            f.write(blob_client.download_blob().readall())
        success += 1
    except Exception as e:
        failed += 1
        if failed <= 5:
            print(f'Failed: {blob_path} — {e}')

print(f'\nDone! Downloaded: {success} | Failed: {failed}')
print(f'Total images in folder: {len(os.listdir(IMAGES_DIR))}')

Total empty annotations: 270849
Preparing to download 3000 images...


100%|████████████████████████████████████████████████████████████████████████████| 3000/3000 [2:18:12<00:00,  2.76s/it]



Done! Downloaded: 3000 | Failed: 0
Total images in folder: 81860
